In [35]:
from os import rename

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker
import importlib
import config
importlib.reload(config)
import pandas_market_calendars as mcal

from config import BB_PRICES_COMPLETE, SENT_TIMESTAMP_CLEANED,US_TICKERS_FINAL_adj, DAILY_SENTIMENT

In [27]:
#open sentiment row and trading day file
#Note pdf = price dataframe
pdf = pd.read_parquet(BB_PRICES_COMPLETE)
ust = pd.read_csv(US_TICKERS_FINAL_adj)
sdf = pd.read_parquet(SENT_TIMESTAMP_CLEANED)

In [18]:
ust.dtypes

ISIN              object
Ticker            object
start_date_s      object
end_date_s        object
start_date_px     object
end_date_px       object
Sector            object
bb_tcm            object
alt_bb_tcm        object
mkt_cap          float64
turnover         float64
bb_tcm_note       object
bb_start_date     object
bb_end_date       object
dtype: object

In [13]:
#create daily aggregation dataframe - this will be the empty shell we intent to fill
#Note ddf = daily dataframe
ddf = pdf[['ISIN', 'date']].drop_duplicates().reset_index(drop=True)

print(f'Number of duplicated pairs: {ddf.duplicated(subset=['ISIN','date']).sum()}')
print(ddf.shape)

Number of duplicated pairs: 0
(3540016, 2)


In [14]:
ddf.rename(columns={'date':'trading_day'}, inplace=True)

In [24]:
#Merge the sector from ust into the pdf based on ISIN
ddf2 = ddf.merge(ust[['ISIN','Sector']], on='ISIN', how='left')
print(f'lenght of ddf: {len(ddf)}')
print(f'lenght of ddf2: {len(ddf)}')
print(ddf2['Sector'].value_counts())
print(ddf2['Sector'].isnull().sum())

lenght of ddf: 3540016
lenght of ddf2: 3540016
Sector
Financials                694889
Consumer Discretionary    629726
Industrials               586687
Information Technology    459119
Health Care               356872
Consumer Staples          193115
Materials                 180992
Utilities                 160297
Energy                    150376
Real Estate                79255
Communication Services     48688
Name: count, dtype: int64
0


In [31]:
sdf.rename(columns={'Isin':'ISIN'}, inplace=True)
sdf.columns

Index(['StoryID', 'Timestamp_utc', 'Ticker', 'Country', 'ISIN', 'Sentiment',
       'Confidence', 'Novelty', 'Topic', 'Relevance', 'HeadlineOnly',
       'AutoMessage', 'Source', 'MarketImpactScore', 'Prob_POS', 'Prob_NTR',
       'Prob_NEG', 'Timestamp'],
      dtype='object')

In [36]:
"""
Aggregates raw sentiment stories down to one row per
(ISIN, trading_day), then attaches sector-relative leave-one-out z-scores and
merges the result onto the trading-day skeleton.

Input dataframes
    sdf  - raw sentiment dataframe
    ddf2 - empty dataframe: ISIN, trading_day, Sector

Output dataframe and new measures.
    sentiment_daily.parquet - one row per (ISIN, trading_day, Sector) with:
        sentiment_score      relevance x confidence weighted mean sentiment
        sentiment_volume     log1p(count of new stories, Novelty == 1)
        sentiment_score_z    leave-one-out z-score vs covered sector peers
        sentiment_volume_z   leave-one-out z-score vs covered sector peers
"""

# ---------------------------------------------------------------------------
# 0. Load inputs if not already loaded
if 'sdf' in globals() and sdf is not None:
    print(f'sdf already loaded in the shape of: {sdf.shape}')
else:
    sdf = pd.read_parquet(SENT_TIMESTAMP_CLEANED)
    sdf.rename(columns={'Isin':'ISIN'}, inplace=True)

# ---------------------------------------------------------------------------
# 1. Story-level preparation
# ---------------------------------------------------------------------------
#Remove the time-zone awareness and set all times to midnight so it's only looking at the specific day date.
sdf['trading_day'] = sdf['Timestamp'].dt.normalize().dt.tz_localize(None)

sdf['weight'] = sdf['Relevance'] * sdf['Confidence']
sdf['weighted_sentiment'] = sdf['weight'] * sdf['Sentiment']

sdf['is_new_story'] = (sdf['Novelty'] == 1).astype('int8')


# ---------------------------------------------------------------------------
# 2. Aggregate to one row per (Isin, trading_day)
# ---------------------------------------------------------------------------
agg = (
    sdf.groupby(['ISIN', 'trading_day'], sort=False)
       .agg(
           weighted_sentiment_sum=('weighted_sentiment', 'sum'),
           weight_sum=('weight', 'sum'),
           new_story_count=('is_new_story', 'sum'),
       )
       .reset_index()
)

# Weighted mean sentiment, with a guard for the rare case where stories exist
# but every weight is zero (all-tied probabilities -> Confidence == 0).
agg['sentiment_score'] = np.where(
    agg['weight_sum'] > 0,
    agg['weighted_sentiment_sum'] / agg['weight_sum'],
    0.0,
)

# Volume of genuinely new coverage, log-compressed.
agg['sentiment_volume'] = np.log1p(agg['new_story_count'])

agg = agg.drop(columns=['weighted_sentiment_sum', 'weight_sum',
                        'new_story_count'])


# ---------------------------------------------------------------------------
# 3. Attach Sector and restrict to the modelling universe
# ---------------------------------------------------------------------------
# Inner merge: keeps only (ISIN, trading_day) combinations that exist in the
# skeleton, i.e. days on which the stock actually traded and is in the
# universe. Sentiment tagged to out-of-universe ISINs or non-skeleton days is
# deliberately dropped here.
agg = agg.merge(ddf2,on=['ISIN', 'trading_day'],how='inner')

# ---------------------------------------------------------------------------
# 4. Leave-one-out sector z-scores (covered stocks only)
# ---------------------------------------------------------------------------
MIN_GROUP_SIZE = 4  # stock itself + at least 3 peers with news that day
def add_loo_zscore(df, value_col, z_col, min_group_size=MIN_GROUP_SIZE):
    """
    Adds a leave-one-out z-score column: how unusual is this stock's value
    relative to the OTHER covered stocks in the same (Sector, trading_day)
    group. Groups smaller than min_group_size get z = 0.
    """
    grp = df.groupby(['Sector', 'trading_day'], sort=False)

    x = df[value_col]
    n = grp[value_col].transform('count')
    s = grp[value_col].transform('sum')

    df['_sq'] = x ** 2
    q = df.groupby(['Sector', 'trading_day'], sort=False)['_sq'].transform('sum')
    df.drop(columns='_sq', inplace=True)

    # Mean of the group excluding the stock itself
    loo_mean = (s - x) / (n - 1)

    # Sample variance (ddof=1) of the group excluding the stock itself:
    #   peers' sum of squares  = q - x^2
    #   peers' count           = n - 1
    #   var = (sum_sq - count * mean^2) / (count - 1)
    loo_var = (q - x ** 2 - (n - 1) * loo_mean ** 2) / (n - 2)

    valid = (n >= min_group_size) & (loo_var > 0)

    df[z_col] = np.where(
        valid,
        (x - loo_mean) / np.sqrt(loo_var.where(valid)),
        0.0,
    )
    return df

# Apply the above function to calculate the sector sentiment z score and sector volume sentiment z score
agg = add_loo_zscore(agg, 'sentiment_score', 'sentiment_score_z')
agg = add_loo_zscore(agg, 'sentiment_volume', 'sentiment_volume_z')


# ---------------------------------------------------------------------------
# 5. Merge onto the full skeleton and zero-fill no-news days
# ---------------------------------------------------------------------------
feature_cols = ['sentiment_score', 'sentiment_volume',
                'sentiment_score_z', 'sentiment_volume_z']

daily = ddf2.merge(
    agg[['ISIN', 'trading_day'] + feature_cols],
    on=['ISIN', 'trading_day'],
    how='left',
)

#Fill any empty columns with a zero.
daily[feature_cols] = daily[feature_cols].fillna(0.0)


# ---------------------------------------------------------------------------
# 6. Sanity checks and save
# ---------------------------------------------------------------------------
assert len(daily) == len(ddf2), 'Row count changed during merge - investigate'
assert not daily.duplicated(subset=['ISIN', 'trading_day']).any(), \
    'Duplicate (ISIN, trading_day) rows found - investigate'

print(f'Skeleton rows          : {len(ddf2):,}')
print(f'Stock-days with news   : {len(agg):,} '
      f'({len(agg) / len(daily):.1%} of skeleton)')
print(f'Non-zero score z-scores: {(daily["sentiment_score_z"] != 0).sum():,}')
print(daily[feature_cols].describe())

daily.to_parquet(DAILY_SENTIMENT, index=False)
print('Saved sentiment_daily.parquet')

sdf already loaded in the shape of: (3546807, 22)
Skeleton rows          : 3,540,016
Stock-days with news   : 795,363 (22.5% of skeleton)
Non-zero score z-scores: 792,782
       sentiment_score  sentiment_volume  sentiment_score_z  \
count     3.540016e+06      3.540016e+06       3.540016e+06   
mean     -7.753485e-03      2.540330e-01       3.390336e-04   
std       2.627857e-01      5.526825e-01       6.879768e-01   
min      -1.000000e+00      0.000000e+00      -2.469411e+02   
25%       0.000000e+00      0.000000e+00       0.000000e+00   
50%       0.000000e+00      0.000000e+00       0.000000e+00   
75%       0.000000e+00      0.000000e+00       0.000000e+00   
max       1.000000e+00      6.006353e+00       5.337835e+02   

       sentiment_volume_z  
count        3.540016e+06  
mean         2.852365e+03  
std          4.365227e+05  
min         -6.441081e+00  
25%          0.000000e+00  
50%          0.000000e+00  
75%          0.000000e+00  
max          1.456161e+08  
Saved sen